In [ ]:
from typing import Any
import pandas as pd
import ast
from pathlib import Path
import glob


def read_csv(path: Path):
    # Read the CSV, skip the first comment line
    df = pd.read_csv(path.resolve(), comment="#")
    # Take the first row and convert it to a dict (columns become keys)
    row_dict = df.iloc[0].to_dict()
    # Try to parse each value that looks like a dictionary
    for key, value in row_dict.items():
        if isinstance(value, str) and value.strip().startswith(("{", "[")):
            try:
                row_dict[key] = ast.literal_eval(value)
            except Exception:
                pass  # Leave as-is if not safely parsable
    return row_dict

In [ ]:
results: dict[str, list[dict[str, Any]]] = {}
loads = {}
orders = {}

for file in glob.glob("*.csv"):
    path = Path(file)
    name = path.name
    order0, order1, alpha, load = name.split("net.evaluations")[0].split("_")

    if not alpha in results:
        results[alpha] = {}
    try:
        csv = read_csv(path)
        order = f"{order0}-{order1}"
        csv["load"] = load
        if order not in results[alpha]:
            results[alpha][order] = []
        results[alpha][order].append(csv)
    except Exception:
        pass

In [ ]:
import matplotlib.pyplot as plt

block_prob: dict[str, list[tuple[str, float]]] = {}


for key, data in results.items():
    plt.figure(figsize=(10, 6))

    for order_key, order_data in data.items():
        if order_key == "ESLC-ESLC":
            continue
        pares = [
            (float(csv["load"]), csv["blockedEvents"] / (csv["steps"]))
            for csv in order_data
        ]

        # Ordenar por el valor de x
        pares_ordenados = sorted(pares, key=lambda p: p[0])

        # Separar los ejes ordenados
        eje_x = [x for x, y in pares_ordenados]
        eje_y = [y for x, y in pares_ordenados]

        plt.plot(eje_x, eje_y, label=order_key)
        plt.legend()
    # labels = [csv["order"] for csv in data if csv[]]
    # plt.scatter(eje_x, eje_y, color="blue")
    plt.margins(y=0.1)

    # Etiquetas y título
    plt.xlabel("Carga de tráfico")
    plt.ylabel(f"Probabilidad de bloqueo para alpha={key}")
    plt.title("Probabilidad de bloqueo vs Carga de tráfico")
    plt.grid(True)
    plt.tight_layout()
    plt.show()